In [ ]:
import zarr
import numpy as np
from matplotlib import pyplot as plt
import os

def get_mfx_mbm(dir_path):
    # Get all folder names from the given directory path 
    folder_names = [name for name in os.listdir(dir_path) if os.path.isdir(os.path.join(dir_path, name))]
    # Create an empaty minflux dictonary

    mfx_dic = {}
    # Loop through folders, get minflux and mbm data and fill the dictonary 
    for folder in folder_names:
        print("Processing folder: ", folder)
        mfx_raw = zarr.load(dir_path +"\\" + folder + '\\mfx' ) # get minflux raw array
        mfx_dic[folder] = {'mfx_raw': mfx_raw, 'mbm': {}, 'rois': {}} # assign minflux array
        mbm_points = zarr.load(dir_path +"\\" + folder + '\\grd\\mbm\\points') # get mbm information
        print(mbm_points)
        mbm_used =  zarr.load(dir_path +"\\" + folder + '\\mbm').grp.attrs['used'] # read the name of the used POR, beads, for the given measurement
        attrs = zarr.load(dir_path  +"\\" + folder).grp.attrs['rois'] # get MINFLUX ROI
        # Check how many MINFLUX ROI are present
        if len(attrs) ==1:
            MFXcorners = np.array(attrs[0]['corners'])
        else:        
            MFXcorners = np.zeros((2,2))
            for i in range(len(attrs)):
                MFXcorners = np.array(attrs[0]['corners'])
                MFXcorners = np.concatenate((MFXcorners, np.array(attrs[i]['corners'])))
                
        # save MINFLUX ROI top left and bottom right corners to MINFLUX dictonary        
        mfx_dic[folder]['ROI'] = MFXcorners 
        # Load mbm information 
        mbm_gri = zarr.load(dir_path +"\\" + folder + '\\grd\\mbm').grp.points.attrs['points_by_gri'] # get mbm information to check wich beads were used in the measurement
        # Loop through mbm grid points
        for key in mbm_gri: 
            pts = mbm_points[mbm_points['gri']==int(key)]
            # Update mbm dictonary with the beads names, grid point, and measurements
            mfx_dic[folder]['mbm'][mbm_gri[key]['name']] = {'bead_name' : mbm_gri[key]['name'],'gri': key, 'used': 0, 'points': pts } 
            # Check which beads were used for the measuremnt correction
            if  mbm_gri[key]['name']in mbm_used:
                mfx_dic[folder]['mbm'][mbm_gri[key]['name']]['used'] = 1
            else:
                mfx_dic[folder]['mbm'][mbm_gri[key]['name']]['used'] = 0
    return mfx_dic

In [13]:
# set the path to the chosen directory, where all the MINFLUX mearurements zarr files from the same exchange PAINT were extracted.   
folder_directory= r"../data/multicolor/"
# use  get_mfx_mbm function to build our MINFLUX dictonary with mbm information 
mfx_dictionary = get_mfx_mbm(folder_directory)

Processing folder:  250320-133207_nup62.minflux
None
{'250320-133207_nup62.minflux': {'mfx_raw': None, 'mbm': {}, 'rois': {}}}
None


AttributeError: 'NoneType' object has no attribute 'grp'